In [ ]:
!pip install rasterio 
!pip install fiona
!pip install rasterstats
!pip install raster4ml
!pip install sklearn
!pip install shapely

In [ ]:
import numpy as np
import fiona
import rasterio.mask
from matplotlib import pyplot
import pandas as pd
import glob
import os
import rasterio
import raster4ml
from rasterio.crs import CRS
from rasterio.plot import show
from raster4ml.extraction import batch_extract_by_polygons, extract_shape_values
import os
import gc
import shutil


os.chdir(r'C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Cahn\CHACABUCO\CHACABUCO\LARGO1\\')

def create_directory(dir, name):
    # Directory
    directory = name

    # Parent Directory path
    parent_dir = dir

    # Path
    path = os.path.join(parent_dir, directory)

    # Create the directory
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)

In [ ]:
def get_bands(pixel_values):
    bands = dict.fromkeys(['blue', 'green', 'red', 'rede', 'nir'], [])
    bands['blue'] = pixel_values[0]
    bands['green'] = pixel_values[1]
    bands['red'] = pixel_values[2]
    bands['rede'] = pixel_values[3]
    bands['nir'] = pixel_values[4]
    return bands

def generate_indexes(dir, profile, bands):
    profile.update(
        dtype=rasterio.float32,
        count=1)
  
    create_directory(dir, "INDEXES")
    with rasterio.open(dir+"/INDEXES/blue.tif", 'w', **profile) as rst:
        rst.write_band(1, bands['blue'].astype('float32'))
        del rst
        gc.collect()

    
    with rasterio.open(dir+"/INDEXES/green.tif", 'w', **profile) as rst:
        rst.write_band(1, bands['green'].astype('float32'))
        del rst
        gc.collect()

    with rasterio.open(dir+"/INDEXES/red.tif", 'w', **profile) as rst:
        rst.write_band(1, bands['red'].astype('float32'))
        del rst
        gc.collect()

    with rasterio.open(dir+"/INDEXES/rede.tif", 'w', **profile) as rst:
        rst.write_band(1, bands['rede'].astype('float32'))
        del rst
        gc.collect()

    with rasterio.open(dir+"/INDEXES/nir.tif", 'w', **profile) as rst:
        rst.write_band(1, bands['nir'].astype('float32'))
        del rst
        gc.collect()

    NDVI = (bands['nir'] - bands['red']) / (bands['nir'] + bands['red'])
    with rasterio.open(dir+"/INDEXES/NDVI.tif", 'w', **profile) as rst:
        rst.write_band(1, NDVI.astype('float32'))
        del rst
        gc.collect()

    GNDVI = (bands['nir']-bands['green'])/(bands['nir']+bands['green'])
    with rasterio.open(dir+"/INDEXES/GNDVI.tif", 'w', **profile) as rst:
        rst.write_band(1, GNDVI.astype('float32'))
        del rst
        gc.collect()

    RVI_1 = bands['nir']/bands['red']
    with rasterio.open(dir+"/INDEXES/RVI_1.tif", 'w', **profile) as rst:
        rst.write_band(1, RVI_1.astype('float32'))
        del rst
        gc.collect()

    GCI = (bands['nir']/bands['green'])-1.0
    with rasterio.open(dir+"/INDEXES/GCI.tif", 'w', **profile) as rst:
        rst.write_band(1, GCI.astype('float32'))
        del rst
        gc.collect()

    RGVI = bands['red']/bands['green']
    with rasterio.open(dir+"/INDEXES/RGVI.tif", 'w', **profile) as rst:
        rst.write_band(1, RGVI.astype('float32'))
        del rst
        gc.collect()

    DVI = bands['nir']-bands['red']
    with rasterio.open(dir+"/INDEXES/DVI.tif", 'w', **profile) as rst:
        rst.write_band(1, DVI.astype('float32'))
        del rst
        gc.collect()
    
    L = 0.5
    SAVI = ((bands['nir']-bands['red'])/(bands['nir']+bands['red']+L))*(1.0+L)
    with rasterio.open(dir+"/INDEXES/SAVI.tif", 'w', **profile) as rst:
        rst.write_band(1, SAVI.astype('float32'))
        del rst
        gc.collect()

    MSAVI = 0.5*((2.0*bands['nir'])+1.0-np.sqrt(np.square(2.0*bands['nir']+1.0)-8.0*(bands['nir']-bands['red'])))
    with rasterio.open(dir+"/INDEXES/MSAVI.tif", 'w', **profile) as rst:
        rst.write_band(1, MSAVI.astype('float32'))
        del rst
        gc.collect()

    OSAVI = (bands['nir']-bands['red'])/(bands['nir']+bands['red']+0.16)
    with rasterio.open(dir+"/INDEXES/OSAVI.tif", 'w', **profile) as rst:
        rst.write_band(1, OSAVI.astype('float32'))
        del rst
        gc.collect()

    RDVI = np.sqrt((np.square(bands['nir']-bands['red']))/(bands['nir']+bands['red']))
    with rasterio.open(dir+"/INDEXES/RDVI.tif", 'w', **profile) as rst:
        rst.write_band(1, RDVI.astype('float32'))
        del rst
        gc.collect()

    TVI = 60.0*(bands['nir']-bands['green'])-100.0*(bands['red']-bands['green'])
    with rasterio.open(dir+"/INDEXES/TVI.tif", 'w', **profile) as rst:
        rst.write_band(1, TVI.astype('float32'))
        del rst
        gc.collect()

    a = 0.96916 
    b = 0.084726
    TSAVI = (a*(bands['nir']-a*bands['red']-b))/(a*bands['nir']+bands['red']-a*b)
    with rasterio.open(dir+"/INDEXES/TSAVI.tif", 'w', **profile) as rst:
        rst.write_band(1, TSAVI.astype('float32'))
        del rst
        gc.collect()

    PVI = (bands['nir']-a*bands['red']-b)/np.sqrt(1+np.square(a))
    with rasterio.open(dir+"/INDEXES/TSAVI.tif", 'w', **profile) as rst:
        rst.write_band(1, TSAVI.astype('float32'))
        del rst
        gc.collect()

    SAVI_2 = bands['nir']/(bands['red']-(b/a))
    with rasterio.open(dir+"/INDEXES/SAVI_2.tif", 'w', **profile) as rst:
        rst.write_band(1, SAVI_2.astype('float32'))
        del rst
        gc.collect()

    X = 0.08
    ATSAVI= (a*(-a*bands['red']-b))/(a*bands['nir']+bands['red']-a*b+X*(1+np.square(a)))
    with rasterio.open(dir+"/INDEXES/ATSAVI.tif", 'w', **profile) as rst:
        rst.write_band(1, ATSAVI.astype('float32'))
        del rst
        gc.collect()

    NDWI = (bands['green']-bands['nir'])/(bands['green']+bands['nir'])
    with rasterio.open(dir+"/INDEXES/NDWI.tif", 'w', **profile) as rst:
        rst.write_band(1, NDWI.astype('float32'))
        del rst
        gc.collect()

    NPCI = (bands['red']-bands['blue'])/(bands['red']+bands['blue'])
    with rasterio.open(dir+"/INDEXES/NPCI.tif", 'w', **profile) as rst:
        rst.write_band(1, NPCI.astype('float32'))
        del rst
        gc.collect()

    SRPI = bands['blue']/bands['red']
    with rasterio.open(dir+"/INDEXES/SRPI.tif", 'w', **profile) as rst:
        rst.write_band(1, SRPI.astype('float32'))
        del rst
        gc.collect()

    RVI_2 = bands['nir']/bands['green']
    with rasterio.open(dir+"/INDEXES/RVI_2.tif", 'w', **profile) as rst:
        rst.write_band(1, RVI_2.astype('float32'))
        del rst
        gc.collect()

    MCARI = (bands['rede']-bands['red']-0.2*(bands['rede']-bands['green']))*(bands['rede']/bands['red'])
    with rasterio.open(dir+"/INDEXES/MCARI.tif", 'w', **profile) as rst:
        rst.write_band(1, MCARI.astype('float32'))
        del rst
        gc.collect()

    MCARI_1 = 1.2*(2.5*(bands['nir']-bands['red'])-1.3*(bands['nir']-bands['green']))
    with rasterio.open(dir+"/INDEXES/MCARI_1.tif", 'w', **profile) as rst:
        rst.write_band(1, MCARI_1.astype('float32'))
        del rst
        gc.collect()

    MCARI_2 = 1.5*(2.5*(bands['nir']-bands['red'])-1.3*(bands['nir']-bands['green']))*(np.square(2.0*bands['nir']+1))-(6.0*bands['nir']-5.0*bands['red'])-0.5
    with rasterio.open(dir+"/INDEXES/MCARI_2.tif", 'w', **profile) as rst:
        rst.write_band(1, MCARI_2.astype('float32'))
        del rst
        gc.collect()

    MTVI_1 = 1.2*(1.2*(bands['nir']-bands['green'])-2.5*(bands['red']-bands['green']))
    with rasterio.open(dir+"/INDEXES/MTVI_1.tif", 'w', **profile) as rst:
        rst.write_band(1, MTVI_1.astype('float32'))
        del rst
        gc.collect()

    MTVI_2 = 1.5*(1.2*(bands['nir']-bands['green'])-2.5*(bands['red']-bands['green']))*(np.square(2*bands['nir']+1))-(6.0*bands['nir']-5.0*bands['red'])-0.5
    with rasterio.open(dir+"/INDEXES/MTVI_2.tif", 'w', **profile) as rst:
        rst.write_band(1, MTVI_2.astype('float32'))
        del rst
        gc.collect()

    R_MCARI_MTVI2 = ((bands['rede']-bands['red']-0.2*(bands['rede']-bands['green']))*(bands['rede']/bands['red']))/(1.5*(1.2*(bands['nir']-bands['green'])-2.5*(bands['red']-bands['green']))*(np.square(2*bands['nir']+1))-(6.0*bands['nir']-5.0*bands['red'])-0.5)
    with rasterio.open(dir+"/INDEXES/R_MCARI_MTVI2.tif", 'w', **profile) as rst:
        rst.write_band(1, R_MCARI_MTVI2.astype('float32'))
        del rst
        gc.collect()

    EVI = (bands['nir']-bands['red'])/(bands['nir']+6.0*bands['red']-7.5*bands['blue']+1.0)
    with rasterio.open(dir+"/INDEXES/EVI.tif", 'w', **profile) as rst:
        rst.write_band(1, EVI.astype('float32'))
        del rst
        gc.collect()

    DATT = (bands['nir']-bands['rede'])/(bands['nir']-bands['red'])
    with rasterio.open(dir+"/INDEXES/DATT.tif", 'w', **profile) as rst:
        rst.write_band(1, DATT.astype('float32'))
        del rst
        gc.collect()

    NDCI = (bands['rede']-bands['green'])/(bands['rede']+bands['green'])
    with rasterio.open(dir+"/INDEXES/NDCI.tif", 'w', **profile) as rst:
        rst.write_band(1, NDCI.astype('float32'))
        del rst
        gc.collect()

    PSRI = (bands['red']-bands['green'])/bands['rede']
    with rasterio.open(dir+"/INDEXES/PSRI.tif", 'w', **profile) as rst:
        rst.write_band(1, PSRI.astype('float32'))
        del rst
        gc.collect()

    SIPI = (bands['nir']-bands['blue'])/(bands['nir']+bands['red'])
    with rasterio.open(dir+"/INDEXES/SIPI.tif", 'w', **profile) as rst:
        rst.write_band(1, SIPI.astype('float32'))
        del rst
        gc.collect()

    SPVI = 0.4*3.7*(bands['nir']-bands['red'])-1.2*np.absolute(bands['green']-bands['red'])
    with rasterio.open(dir+"/INDEXES/SPVI.tif", 'w', **profile) as rst:
        rst.write_band(1, SPVI.astype('float32'))
        del rst
        gc.collect()

    TCARI = 3.0*((bands['rede']-bands['red'])-0.2*(bands['rede']-bands['green'])*(bands['rede']/bands['red']))
    with rasterio.open(dir+"/INDEXES/TCARI.tif", 'w', **profile) as rst:
        rst.write_band(1, TCARI.astype('float32'))
        del rst
        gc.collect()

    R_TCARI_OSAVI = (3.0*((bands['rede']-bands['red'])-0.2*(bands['rede']-bands['green'])*(bands['rede']/bands['red'])))/((bands['nir']-bands['red'])/(bands['nir']+bands['red']+0.16))
    with rasterio.open(dir+"/INDEXES/R_TCARI_OSAVI.tif", 'w', **profile) as rst:
        rst.write_band(1, R_TCARI_OSAVI.astype('float32'))
        del rst
        gc.collect()

    RERI = (bands['rede']-bands['red'])/bands['nir']
    with rasterio.open(dir+"/INDEXES/RERI.tif", 'w', **profile) as rst:
        rst.write_band(1, RERI.astype('float32'))
        del rst
        gc.collect()

    NDRE = (bands['nir']-bands['rede'])/(bands['nir']+bands['rede'])
    with rasterio.open(dir+"/INDEXES/NDRE.tif", 'w', **profile) as rst:
        rst.write_band(1, NDRE.astype('float32'))
        del rst
        gc.collect()

    MTCI = (bands['nir']-bands['rede'])/(bands['rede']-bands['red'])
    with rasterio.open(dir+"/INDEXES/MTCI.tif", 'w', **profile) as rst:
        rst.write_band(1, MTCI.astype('float32'))
        del rst
        gc.collect()

    EVI_2 = 2.5*((bands['nir']-bands['red'])/(bands['nir']+2.4*bands['red']+1.0))
    with rasterio.open(dir+"/INDEXES/EVI_2.tif", 'w', **profile) as rst:
        rst.write_band(1, EVI_2.astype('float32'))
        del rst
        gc.collect()

    RECI = (bands['nir']/bands['rede'])-1
    with rasterio.open(dir+"/INDEXES/RECI.tif", 'w', **profile) as rst:
        rst.write_band(1, RECI.astype('float32'))
        del rst
        gc.collect()

    NEXG = (2*bands['green']-bands['red']-bands['blue'])/(bands['green']+bands['red']+bands['blue'])
    with rasterio.open(dir+"/INDEXES/NEXG.tif", 'w', **profile) as rst:
        rst.write_band(1, NEXG.astype('float32'))
        del rst
        gc.collect()

    NGRDI = (bands['green']-bands['red'])/(bands['green']+bands['red'])
    with rasterio.open(dir+"/INDEXES/NGRDI.tif", 'w', **profile) as rst:
        rst.write_band(1, NGRDI.astype('float32'))
        del rst
        gc.collect()

    ENDVI = (bands['nir']+bands['green']-2.0*bands['blue'])/(bands['nir']+bands['green']+2.0*bands['blue'])
    with rasterio.open(dir+"/INDEXES/ENDVI.tif", 'w', **profile) as rst:
        rst.write_band(1, ENDVI.astype('float32'))
        del rst
        gc.collect()

    ARI_2 = bands['nir']*((1.0/bands['green'])-(1.0/bands['rede']))
    with rasterio.open(dir+"/INDEXES/ARI_2.tif", 'w', **profile) as rst:
        rst.write_band(1, ARI_2.astype('float32'))
        del rst
        gc.collect()

    CRI_2 = (1.0/bands['green'])-(1.0/bands['rede'])
    with rasterio.open(dir+"/INDEXES/CRI_2.tif", 'w', **profile) as rst:
        rst.write_band(1, CRI_2.astype('float32'))
        del rst
        gc.collect()

    #HERE WE CAN VISUALIZE THE INDEX MAPS
    #pyplot.imshow(NDVI)
    #pyplot.show()

    values = batch_extract_by_polygons(dir+"/INDEXES", 
                                       r"C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Cahn\CHACABUCO\CHACABUCO\LARGO1\PLOTBOUNDARIES\PLOTBOUNDARIES_PROJECT.shp",
                                       "ID", statistics=['mean'])
    create_directory(dir, "VALUES")
    
    disease_labels = extract_shape_values(r"C:\Users\flopes1\OneDrive - Saint Louis University\Desktop\Cahn\CHACABUCO\CHACABUCO\LARGO1\PLOTBOUNDARIES\PLOTBOUNDARIES_PROJECT.shp",
                                        "ID", columns=['name', 'HEALTH_STA'])

    # Adding the column "health status" to the statistic summary
    values['HEALTH_STA'] = disease_labels['HEALTH_STA']
    values.to_csv(dir+"/VALUES/values.csv")
    
    # The code above calculates indexes and create a raster for each of them, then saves them into a folder

In [ ]:
def normalize_pixels(pixel_values):
    pixel_values = (pixel_values - pixel_values.min()) / (pixel_values.max() - pixel_values.min()) 
    return pixel_values

"""
Considering the Chacabuco field and its 6 dates dataset
1. For each date:
1.1 Open the raster images
1.2 Get the pixels
1.3 Normalize pixel values
1.4 Get de bands values
1.5 Save specifc bands .tif images
1.6 Calculate indexes
"""
def main():
    dir_path = './'
    ignore_dir = "INDEXES"
    # Walks through the subdirectories and list files with .tif extension
    for root, dirs, files in os.walk(dir_path):
        print(dirs)
        if ignore_dir in dirs:
            dirs.remove("INDEXES")
            continue

        for file in files:
            if file.endswith(".tif"):
                file_path = os.path.join(root, file)
                # Open the file here
                with rasterio.open(file_path) as f:
                    print(f"File {file_path} opened with success!")

                    pixel_values = f.read()
                    profile = f.profile
                    del f
                    gc.collect()

                    pixel_values = normalize_pixels(pixel_values)
                    bands = get_bands(pixel_values)
                    del pixel_values
                    gc.collect()

                    generate_indexes(root, profile, bands)
                    del profile
                    del bands
                    gc.collect()

                    """
                    image_norm = (pixel_values - pixel_values.min()) / (pixel_values.max() - pixel_values.min()) 
                    del image_norm
                    gc.collect()

                    Now we can use the show function from rasterio, passing in the image to display it.
                    Note that this function expects the numpy array to be either a float ranging from 0 to 1, or an uint8 ranging from 0 to 255. 
                    Since our image is an uint16, we first normalize in order for it to render properly.
                    show(image_norm)
                """
main()

In [ ]:
!pip install scikit-learn
!pip install seaborn
!pip install imblearn

In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.svm import SVR
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, precision_recall_fscore_support
from sklearn import preprocessing
from sklearn.model_selection import StratifiedShuffleSplit

from datetime import datetime

import geopandas as gpd
import numpy as np
import glob


# Specify the directory where the CSV files are located
directory = '/Users/felipealencar/Desktop/plant-disease-prediction/dataset/LARGO1/'

# Use the glob module to retrieve a list of file paths that match the pattern '*.csv' in the specified directory and its subdirectories
file_list = glob.glob(directory + '/**/*.csv', recursive=True)

# Create an empty dataframe to store the concatenated data
df = pd.DataFrame()

# Loop through the file list and read each CSV file using pd.read_csv(), and concatenate it to the dataframe
for file in file_list:
    print(file)
    df_temp = pd.read_csv(file)
    # Extract the date from the filename (assuming the filename has a date string in it)
    date_str = file.split('_')[2].split('/')[0] # assuming the date is the first part of the filename
    date_obj = datetime.strptime(date_str, '%m%d')
    desired_year = 2021
    correct_date_obj = date_obj.replace(year=desired_year)

    formatted_date_str = correct_date_obj.strftime('%m/%d/%Y')
    print(formatted_date_str)
    date = pd.to_datetime(formatted_date_str).date()
    
    # Add a new column for the date
    df_temp['date'] = date

    df = pd.concat([df, df_temp], ignore_index=True)

#df = pd.read_csv(r"/Users/felipealencar/Desktop/plant-disease-prediction/dataset/LARGO1/LARGO_1_0830/values.csv") 
df_next_date = pd.read_csv(r"/Users/felipealencar/Desktop/plant-disease-prediction/dataset/LARGO_1_1117/values.csv")

"""
date_str = '0830'
date_obj = datetime.strptime(date_str, '%m%d')
desired_year = 2021
correct_date_obj = date_obj.replace(year=desired_year)
formatted_date_str = correct_date_obj.strftime('%m/%d/%Y')
print(formatted_date_str)
date = pd.to_datetime(formatted_date_str).date()
df['date'] = date
"""

date_str = '1117'
date_obj = datetime.strptime(date_str, '%m%d')
desired_year = 2021
correct_date_obj = date_obj.replace(year=desired_year)
formatted_date_str = correct_date_obj.strftime('%m/%d/%Y')
print(formatted_date_str)
date = pd.to_datetime(formatted_date_str).date()
df_next_date['date'] = date

df = df[df['HEALTH_STA'].notna()]
le = preprocessing.LabelEncoder()
le.fit(df['HEALTH_STA'])
df['HEALTH_STA'] = le.transform(df['HEALTH_STA'])

df_next_date = df_next_date[df_next_date['HEALTH_STA'].notna()]
le = preprocessing.LabelEncoder()
le.fit(df_next_date['HEALTH_STA'])
df_next_date['HEALTH_STA'] = le.transform(df_next_date['HEALTH_STA'])

## Feature & Target Selection
X = df[[col for col in df.columns]]
X = X.drop('HEALTH_STA', axis=1)
X = X.drop('date', axis=1)
X = X.drop(X.columns[0], axis=1)
y = df['HEALTH_STA']
y = y.dropna()
labels = y.unique()

X.replace([np.inf, -np.inf], np.nan, inplace=True)
mean_values = X.mean(axis=0)
X.fillna(mean_values, inplace=True)

min_max_scaler = preprocessing.MinMaxScaler()
x_scaled = min_max_scaler.fit_transform(X)
X = pd.DataFrame(x_scaled)

X = X.astype(float)

from imblearn.over_sampling import RandomOverSampler

# X_train and y_train are your feature matrix and target labels respectively

class_counts = np.bincount(y)
print(class_counts)

majority_class_size = np.max(class_counts)
sampling_strategy = {i: majority_class_size*1 for i in range(len(class_counts)) if class_counts[i] < majority_class_size}

# create an instance of the RandomOverSampler class
ros = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=42)

X_resampled, y_resampled = ros.fit_resample(X, y)

# Split the train and test sets into train and test sets
X_train_test, X_val, y_train_test, y_val = train_test_split(X_resampled, y_resampled, test_size=0.25, stratify=y_resampled, random_state=42)

# Split the train and test sets into train and test sets again
X_train, X_test, y_train, y_test = train_test_split(X_train_test, y_train_test, test_size=0.2, stratify=y_train_test, random_state=42)

counts = np.bincount(y_train)
print(counts)

X_next_date = df_next_date[[col for col in df_next_date.columns]]
X_next_date = X_next_date.drop('HEALTH_STA', axis=1)
X_next_date = X_next_date.drop('date', axis=1)
X_next_date = X_next_date.drop(X_next_date.columns[0], axis=1)
y_next_date = df_next_date['HEALTH_STA']
y_next_date = y_next_date.dropna()

X_next_date.replace([np.inf, -np.inf], np.nan, inplace=True)
next_date_mean_values = X_next_date.mean(axis=0)
X_next_date.fillna(next_date_mean_values, inplace=True)

min_max_scaler = preprocessing.MinMaxScaler()
x_scaled = min_max_scaler.fit_transform(X_next_date)
X_next_date = pd.DataFrame(x_scaled)

X_next_date = X_next_date.astype(float)

class_counts = np.bincount(y_next_date)
print(class_counts)

from imblearn.over_sampling import RandomOverSampler

# X_train and y_train are your feature matrix and target labels respectively

majority_class_size = np.max(class_counts)
sampling_strategy = {i: majority_class_size*1 for i in range(len(class_counts)) if class_counts[i] < majority_class_size}

# create an instance of the RandomOverSampler class
ros = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=42)

X_next_date_resampled, y_next_date_resampled = ros.fit_resample(X_next_date, y_next_date)

# Split the train and test sets into train and test sets
X_train_test, X_val, y_train_test, y_val = train_test_split(X_next_date_resampled, y_next_date_resampled, test_size=0.25, stratify=y_next_date_resampled, random_state=42)

# Split the train and test sets into train and test sets again
X_next_date_train, X_next_date_test, y_next_date_train, y_next_date_test = train_test_split(X_train_test, y_train_test, test_size=0.2, stratify=y_train_test, random_state=42)

class_counts = np.bincount(y_next_date_train)
print(class_counts)


# preview train & test sets
print('Train Set:', X_train.shape, y_train.shape)
print('Test Set:', X_val.shape, y_val.shape)
 
# build, train, & predict model
model = SVC()
model.fit(X_train, y_train)
y_pred = model.predict(X_next_date_test)

# evaluate results
print('SVC Metrics')
f1 = f1_score(y_next_date_test, y_pred, average='micro')
accuracy_score = model.score(X_next_date_test, y_next_date_test)
print('MAE:', mean_absolute_error(y_pred, y_next_date_test))
print('MSE:', mean_squared_error(y_pred, y_next_date_test))
print('R2 Score:', r2_score(y_pred, y_next_date_test))
print('F1-SCORE: ', f1)
print('ACCURACY SCORE:', accuracy_score)

# Calculate accuracy per label
label_accuracy = precision_recall_fscore_support(y_next_date_test, y_pred, average=None)
# The 'average=None' argument returns precision, recall, and F1-score for each label separately

# Print the accuracy per label
for label, accuracy in zip(labels, label_accuracy[0]):
    print(f"Label {label}: Accuracy = {accuracy}")

corr = df.corr()

import seaborn as sns
sns.heatmap(corr)

import numpy as np

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
param_grid = { 
    'n_estimators': [200, 500],
    'max_features': [1, 'sqrt', 'log2'],
    'max_depth' : [4,5,6,7,8],
    'criterion' :['entropy', 'gini']
}

from scipy.stats import uniform

#print('Random Forest Classifier')
#rfc = RandomForestClassifier(random_state=42)
#CV_rfc = GridSearchCV(estimator=rfc, param_grid=param_grid, cv= 5)

# fit
#CV_rfc.fit(X_train, y_train)
#print('Best Params Grid Search:')
#print(CV_rfc.best_params_)

# Instantiate RandomizedSearchCV model
"""sumary_line

Keyword arguments:
argument -- description
Return: return_description
"""
"""
rs_model = RandomizedSearchCV(RandomForestClassifier(n_jobs=-1, random_state=25),
                               param_distributions=param_grid,
                               n_iter=3,
                               cv=2,
                               verbose=True)
rs_model.fit(X_train, y_train)
print('Best Params RandomizedSearch (RF):')
print(rs_model.best_params_)
best_params = rs_model.best_params_

print('Metrics (RFO)')
rfc_best = RandomForestClassifier(**best_params)
rfc_best.fit(X_train, y_train)
y_pred = rfc_best.predict(X_next_date_test)
f1 = f1_score(y_next_date_test, y_pred, average='micro')
accuracy_score = model.score(X_next_date_test, y_next_date_test)
print('MAE:', mean_absolute_error(y_pred, y_next_date_test))
print('MSE:', mean_squared_error(y_pred, y_next_date_test))
print('R2 Score:', r2_score(y_pred, y_next_date_test))
print('F1-SCORE: ', f1)
print('ACCURACY SCORE:', accuracy_score)

# Calculate accuracy per label
label_accuracy = precision_recall_fscore_support(y_next_date_test, y_pred, average=None)
# The 'average=None' argument returns precision, recall, and F1-score for each label separately

# Print the accuracy per label
for label, accuracy in zip(labels, label_accuracy[0]):
    print(f"Label {label}: Accuracy = {accuracy}")
"""

In [ ]:
# Plot the confusion matrix for the SVM model
from sklearn.metrics import plot_confusion_matrix

class_names = ['Healthy', 'Unhealthy', 'Average']

svm_disp = plot_confusion_matrix(model, X_next_date_test, y_next_date_test, cmap=plt.cm.Blues, display_labels=class_names)
svm_disp.ax_.set_title("Confusion matrix for SVM")

# Plot the confusion matrix for the Random Forest model
rf_disp = plot_confusion_matrix(rfc_best, X_next_date_test, y_next_date_test, cmap=plt.cm.Blues, display_labels=class_names)
rf_disp.ax_.set_title("Confusion matrix for Random Forest")

plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

fig, ax = plt.subplots(1, 1, figsize=(20, 20), dpi=70)
list_feature_importance = list(rfc_best.feature_importances_)
list_feature_importance

fig.suptitle('Random Forest Feature Importance', fontsize = 20)
ax.set_ylabel('Features',  fontsize = 18)
ax.set_xlabel('Feature Importance', fontsize = 18)
plt.figure(figsize=(20, 5))

print('Columns')
X.columns = [col for col in df.columns if col != 'HEALTH_STA' and col != 'Unnamed: 0']
ax.barh(y=X.columns, width=list_feature_importance)
importance_dict = dict(zip(X.columns, list_feature_importance))
sorted_labels = sorted(importance_dict, key=lambda k: importance_dict[k])
ax.set_yticklabels(sorted_labels, fontsize=14)
plt.tight_layout()
plt.show()

perm_importance = permutation_importance(rs_model, X_next_date_train, y_next_date_train)
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
sorted_idx = perm_importance.importances_mean.argsort()
fig.suptitle('Permutation Importance', fontsize = 20)
plt.bar(np.array(X.columns)[sorted_idx], perm_importance.importances_mean[sorted_idx])
ax.set_xlabel('Feature Importance', fontsize = 18)
ax.set_xticklabels(np.array(X.columns)[sorted_idx], rotation = 90)

y

# Add confusion matrix to check real accuracy
# Create the confusion matrix
cm = confusion_matrix(y_next_date_test, y_pred)
target_names = y.unique()
# Plot the confusion matrix
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(cm, cmap='Blues')
ax.grid(False)
ax.set_xlabel('Predicted Labels', fontsize = 12, color = 'black')
ax.set_ylabel('True Labels', fontsize = 12, color = 'black')
ax.xaxis.set(ticks=np.arange(len(target_names)))
ax.yaxis.set(ticks=np.arange(len(target_names)))
ax.set_xticklabels(target_names, fontsize = 12, color = 'black')
ax.set_yticklabels(target_names, fontsize = 12, color = 'black')

# Loop over data dimensions and create text annotations.
for i in range(len(target_names)):
    for j in range(len(target_names)):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white", fontsize=14)

plt.show()

# Test the dataset for unhealthy
# Test using other dates as input

In [ ]:
# Extract the dates column and remove duplicates
dates = df['date'].unique()
print(dates)
# Set the dates column as the index
df.set_index('date', inplace=True)


In [18]:
!pip install statsmodels

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 10.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.8/233.8 kB 9.3 MB/s eta 0:00:00


In [19]:
from statsmodels.tsa.arima.model import ARIMA

# Define the time series values as a separate dataframe
ts_data = df.drop('HEALTH_STA', axis=1)
ts_data = ts_data.drop(ts_data.columns[0], axis=1)

# Split the data into training and testing sets
train, test = train_test_split(ts_data, test_size=0.2, random_state=42)

# Fit the ARIMA model
model = ARIMA(train, order=(1, 1, 1))
model_fit = model.fit()

# Forecast the next period
forecast = model_fit.forecast()

print("Forecasted value for the next period:", forecast[0])


/Users/felipealencar/opt/anaconda3/envs/pythondev/lib/python3.7/site-packages/statsmodels/tsa/base/tsa_model.py:471: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Users/felipealencar/opt/anaconda3/envs/pythondev/lib/python3.7/site-packages/statsmodels/tsa/base/tsa_model.py:471: ValueWarning: A date index has been provided, but it is not monotonic and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


ValueError: SARIMAX models require univariate `endog`. Got shape (790, 48).